#  Build silver_asset_reference

We split the mixed bronze reference table into clean Silver lookup subsets.

We only keep:
- Asset reference data
- Clean asset_id + region + asset_type
- Remove irrelevant NULL rows

In [0]:
import yaml
from pyspark.sql import functions as F

In [0]:
config_path = "/Workspace/Repos/Mini Projects/Vattenfall-Engineering-Capstone-Project/config/project_config.yml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

catalog = config["catalog"]
raw_schema = config["schemas"]["raw"]
refined_schema = config["schemas"]["refined"]

In [0]:
df_ref = spark.table(f"{catalog}.{raw_schema}.bronze_reference")

display(df_ref)

In [0]:
df_asset = df_ref.filter(F.col("asset_id").isNotNull())

In [0]:
df_asset = (
    df_asset
    .withColumn("asset_id", F.upper("asset_id"))
    .withColumn("region", F.upper("region"))
    .withColumn("asset_type", F.upper("asset_type"))
)

In [0]:
df_asset = (
    df_ref
    .filter(F.col("asset_id").isNotNull())
    .withColumn("asset_id", F.upper("asset_id"))
    .withColumn("region", F.upper("region"))
    .withColumn("asset_type", F.upper("asset_type"))
    .select(
        "asset_id",
        "asset_name",
        "region",
        "asset_type"
    )
    .dropDuplicates(["asset_id"])
)

In [0]:
display(df_asset)
df_asset.printSchema()

print("Row count:", df_asset.count())

In [0]:
df_asset.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{refined_schema}.silver_asset_reference")

In [0]:
display(
    spark.table(f"{catalog}.{refined_schema}.silver_asset_reference")
)

# Why only 4 columns are present?

The Bronze reference table contains multiple datasets mixed together (asset, market, region, etc.).

Each row only has values for its own domain, while other columns are NULL.

In Silver layer, we kept only asset-related fields using:

- asset_id
- asset_name
- region
- asset_type

So all other unrelated columns were removed.

# Result
Now we have a clean, domain-specific lookup table ready for joins.